In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib seaborn streamlit joblib
# Install cloudflared to run Streamlit in Colab
!wget -q -c https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

Selecting previously unselected package cloudflared.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.11.1) ...
Setting up cloudflared (2026.11.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [ ]:
from google.colab import files
import io

print("Please upload your 'NIDS_Network_Benchmark_APT_2025.csv' file:")
uploaded = files.upload()

filename = list(uploaded.keys())[0]
print(f"Uploaded: {filename}")

Please upload your 'NIDS_Network_Benchmark_APT_2025.csv' file:


Saving NIDS_Bench_2026_dataset.zip to NIDS_Bench_2026_dataset.zip
Uploaded: NIDS_Bench_2026_dataset.zip


In [ ]:
import zipfile

# Unzip the uploaded file
with zipfile.ZipFile("NIDS_Bench_2026_dataset.zip", 'r') as zip_ref:
    zip_ref.extractall(".") # Extracts to the current directory

print("Extraction complete!")

Extraction complete!


In [ ]:
import os
# List all files in the current directory
print(os.listdir())

['.config', 'cloudflared-linux-amd64.deb', 'nids_bench_2026', 'NIDS_Bench_2026_dataset.zip', 'sample_data']


In [ ]:
import os
# Look inside the extracted folder
print(os.listdir('nids_bench_2026'))

['DATASHEET.md', 'CITATION.cff', 'benchmarks', 'scripts', 'figures', 'README.md', 'data', 'FEATURES.md', 'metadata.json', 'LICENSE']


In [ ]:
import os
print(os.listdir('nids_bench_2026/data'))

['SHA256SUMS.txt', 'NIDS_Bench_2026_test.csv', 'NIDS_Bench_2026_val.csv']


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# 1. Load Data
file_path = "nids_bench_2026/data/NIDS_Bench_2026_test.csv"
df = pd.read_csv(file_path)
print(f"Original shape: {df.shape}")

# 2. Data Cleaning
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

# 3. Target Identification
target_col = 'Label' if 'Label' in df.columns else df.select_dtypes(include=['object']).columns[-1]
print(f"Target column identified as: {target_col}")

# 4. Feature Selection (Correlation Filter > 0.95)
X = df.drop(columns=[target_col])
y = df[target_col]

numeric_cols = X.select_dtypes(include=[np.number]).columns
corr_matrix = X[numeric_cols].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]
X_reduced = X.drop(columns=to_drop)
print(f"Features dropped due to high correlation (>0.95): {len(to_drop)}")
print(f"Reduced Feature Count: {X_reduced.shape[1]}")

# 5. Encoding & Scaling
categorical_cols = X_reduced.select_dtypes(include=['object']).columns
for col in categorical_cols:
    le = LabelEncoder()
    X_reduced[col] = le.fit_transform(X_reduced[col])

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_reduced)

# 6. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)
print(f"Training set: {X_train.shape}, Testing set: {X_test.shape}")

Original shape: (120000, 71)
Target column identified as: label
Features dropped due to high correlation (>0.95): 1
Reduced Feature Count: 69
Training set: (96000, 69), Testing set: (24000, 69)


In [ ]:
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import joblib

# --- Random Forest (Multi-class Classification) ---
print("Training Random Forest...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

print("\n--- Random Forest Evaluation ---")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_rf, average='macro'):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_rf, average='macro'):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_rf, average='macro'):.4f}")

# --- Isolation Forest (Anomaly Detection) ---
benign_idx = np.where(label_encoder.classes_ == 'BENIGN')[0][0] if 'BENIGN' in label_encoder.classes_ else 0
X_train_benign = X_train[y_train == benign_idx]

print("\nTraining Isolation Forest on Benign Traffic...")
iso_forest = IsolationForest(contamination=0.05, random_state=42, n_jobs=-1)
iso_forest.fit(X_train_benign)

# Save everything for the Streamlit app
joblib.dump(rf_model, 'rf_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(label_encoder, 'label_encoder.pkl')
joblib.dump(list(X_reduced.columns), 'feature_names.pkl')

# Save the background traffic for the live simulation dashboard
X_test_df = pd.DataFrame(X_test, columns=X_reduced.columns)
X_test_df.to_csv('live_traffic_simulation.csv', index=False)
print("\n✅ Models and simulation data saved!")

Training Random Forest...

--- Random Forest Evaluation ---
Accuracy:  0.9776
Precision: 0.9291
Recall:    0.9266
F1-Score:  0.9242

Training Isolation Forest on Benign Traffic...

✅ Models and simulation data saved!


In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import time
import random
import plotly.express as px
import plotly.graph_objects as go

# --- PAGE CONFIGURATION ---
st.set_page_config(page_title="Sentinel AI | Threat Scanner", layout="wide", page_icon="🛡️")

# --- PREMIUM CUSTOM CSS ---
st.markdown("""
    <style>
    /* Main Background */
    .stApp { background-color: #0E1117; color: #FAFAFA; }

    /* Headers */
    .main-header { font-size: 50px; font-weight: 900; background: -webkit-linear-gradient(45deg, #00C9FF, #92FE9D); -webkit-background-clip: text; -webkit-text-fill-color: transparent; text-align: center; margin-bottom: 0px;}
    .sub-header { font-size: 20px; color: #A0AAB5; text-align: center; margin-bottom: 30px;}

    /* Alert Cards */
    .alert-critical { background: linear-gradient(135deg, #FF416C 0%, #FF4B2B 100%); color: white; padding: 30px; border-radius: 20px; text-align: center; font-size: 32px; font-weight: bold; box-shadow: 0 10px 20px rgba(255, 65, 108, 0.4); animation: pulse 2s infinite;}
    .alert-high { background: linear-gradient(135deg, #F7971E 0%, #FFD200 100%); color: #333; padding: 30px; border-radius: 20px; text-align: center; font-size: 32px; font-weight: bold; box-shadow: 0 10px 20px rgba(247, 151, 30, 0.4);}
    .alert-low { background: linear-gradient(135deg, #11998e 0%, #38ef7d 100%); color: white; padding: 30px; border-radius: 20px; text-align: center; font-size: 32px; font-weight: bold; box-shadow: 0 10px 20px rgba(56, 239, 125, 0.4);}
    .alert-suspicious { background: linear-gradient(135deg, #4CA1AF 0%, #C4E0E5 100%); color: #333; padding: 30px; border-radius: 20px; text-align: center; font-size: 32px; font-weight: bold; box-shadow: 0 10px 20px rgba(76, 161, 175, 0.4);}

    /* Pulse Animation for Critical */
    @keyframes pulse {
        0% { transform: scale(1); box-shadow: 0 0 0 0 rgba(255, 65, 108, 0.7); }
        70% { transform: scale(1.02); box-shadow: 0 0 0 20px rgba(255, 65, 108, 0); }
        100% { transform: scale(1); box-shadow: 0 0 0 0 rgba(255, 65, 108, 0); }
    }

    /* KPI Metric Cards */
    .metric-card { background: #1E2329; padding: 20px; border-radius: 15px; border-left: 5px solid #00C9FF; box-shadow: 0 4px 6px rgba(0,0,0,0.3); text-align: center;}
    .metric-value { font-size: 28px; font-weight: bold; color: #00C9FF; }
    .metric-label { font-size: 14px; color: #A0AAB5; text-transform: uppercase; letter-spacing: 1px;}
    </style>
""", unsafe_allow_html=True)

# --- MODEL LOADING CACHE ---
@st.cache_resource
def load_models():
    return joblib.load('rf_model.pkl'), joblib.load('scaler.pkl'), joblib.load('label_encoder.pkl'), joblib.load('feature_names.pkl')

@st.cache_data
def load_background_traffic():
    try:
        df = pd.read_csv("nids_bench_2026/data/NIDS_Bench_2026_test.csv")
    except:
        df = pd.read_csv("NIDS_Bench_2026_test.csv")
    target_col = 'label' if 'label' in df.columns else df.select_dtypes(include=['object']).columns[-1]
    return df, target_col

try:
    rf_model, scaler, label_encoder, feature_names = load_models()
    live_data_full, target_col = load_background_traffic()
except Exception as e:
    st.error(f"Error loading models or data: {e}. Please ensure all files exist.")
    st.stop()

# --- UI HEADER ---
st.markdown("<h1 class='main-header'>Sentinel AI Threat Scanner</h1>", unsafe_allow_html=True)
st.markdown("<p class='sub-header'>Next-Generation Network Anomaly Detection & Classification</p>", unsafe_allow_html=True)

# --- SIDEBAR CONFIGURATION ---
st.sidebar.markdown("## ⚙️ System Controls")
confidence_threshold = st.sidebar.slider("Threat Sensitivity (Confidence %)", min_value=50, max_value=100, value=80, step=5)
st.sidebar.markdown("---")
st.sidebar.success("🟢 **System Status:** Online\n\n🧠 **Model:** Random Forest\n\n📊 **Dataset:** NIDS_Bench_2026\n\n🔒 **Encryption:** AES-256 Enabled")

# --- SESSION STATE INITIALIZATION ---
if 'history' not in st.session_state: st.session_state.history = []
if 'is_scanning' not in st.session_state: st.session_state.is_scanning = False
if 'chart_data' not in st.session_state: st.session_state.chart_data = pd.DataFrame(columns=['Time', 'Confidence', 'Threat'])
if 'total_scans' not in st.session_state: st.session_state.total_scans = 0
if 'threat_count' not in st.session_state: st.session_state.threat_count = 0

# --- KPI METRICS ROW ---
st.markdown("### 📈 Real-Time Telemetry")
kpi1, kpi2, kpi3, kpi4 = st.columns(4)
kpi1_placeholder = kpi1.empty()
kpi2_placeholder = kpi2.empty()
kpi3_placeholder = kpi3.empty()
kpi4_placeholder = kpi4.empty()

# --- CONTROL BUTTONS ---
col1, col2, col3 = st.columns([1,1,2])
with col1:
    if st.button("▶️ Initialize Scan", use_container_width=True, type="primary"): st.session_state.is_scanning = True
with col2:
    if st.button("⏹️ Terminate", use_container_width=True): st.session_state.is_scanning = False
with col3:
    if st.button("🗑️ Reset Dashboard", use_container_width=True):
        st.session_state.history = []
        st.session_state.chart_data = pd.DataFrame(columns=['Time', 'Confidence', 'Threat'])
        st.session_state.total_scans = 0
        st.session_state.threat_count = 0
        st.rerun()

st.markdown("---")

# --- LAYOUT PLACEHOLDERS ---
alert_placeholder = st.empty()

col_chart, col_pie = st.columns([2, 1])
with col_chart:
    st.markdown("### 📊 Confidence Trend Analysis")
    chart_placeholder = st.empty()
with col_pie:
    st.markdown("### 🎯 Threat Distribution")
    pie_placeholder = st.empty()

st.markdown("### 📜 Live Threat Log")
table_placeholder = st.empty()

# Initial rendering of empty KPI cards
def update_kpis(total, threats, avg_conf):
    kpi1_placeholder.markdown(f"<div class='metric-card'><div class='metric-value'>{total}</div><div class='metric-label'>Total Packets Scanned</div></div>", unsafe_allow_html=True)
    kpi2_placeholder.markdown(f"<div class='metric-card' style='border-left-color: #FF4B2B;'><div class='metric-value' style='color:#FF4B2B;'>{threats}</div><div class='metric-label'>Active Threats</div></div>", unsafe_allow_html=True)
    kpi3_placeholder.markdown(f"<div class='metric-card' style='border-left-color: #38ef7d;'><div class='metric-value' style='color:#38ef7d;'>{avg_conf:.1f}%</div><div class='metric-label'>Avg Confidence</div></div>", unsafe_allow_html=True)
    kpi4_placeholder.markdown(f"<div class='metric-card' style='border-left-color: #F7971E;'><div class='metric-value' style='color:#F7971E;'>Active</div><div class='metric-label'>System Status</div></div>", unsafe_allow_html=True)

update_kpis(0, 0, 0.0)

# Get unique attack classes for balanced simulation
unique_labels = live_data_full[target_col].unique()

# --- LIVE SCANNING LOOP ---
if st.session_state.is_scanning:
    while st.session_state.is_scanning:
        # 1. Simulate Traffic
        random_class = random.choice(unique_labels)
        class_subset = live_data_full[live_data_full[target_col] == random_class]
        random_idx = random.randint(0, len(class_subset) - 1)
        raw_flow_full = class_subset.iloc[[random_idx]]

        raw_flow_features = raw_flow_full.drop(columns=[target_col])
        actual_label = raw_flow_full[target_col].values[0]

        raw_flow_features = raw_flow_features.reindex(columns=feature_names, fill_value=0)
        for col in raw_flow_features.columns:
            if raw_flow_features[col].dtype == 'object':
                try: raw_flow_features[col] = raw_flow_features[col].astype(float)
                except ValueError: raw_flow_features[col] = raw_flow_features[col].apply(lambda x: hash(str(x)) % 1000)

        # 2. Predict
        X_scaled = scaler.transform(raw_flow_features)
        pred_idx = rf_model.predict(X_scaled)[0]
        pred_prob = np.max(rf_model.predict_proba(X_scaled)) * 100
        threat_type = label_encoder.inverse_transform([pred_idx])[0]

        # 3. Threat Logic
        if threat_type.lower() == 'benign':
            threat_level, css_class, icon = "BENIGN", "alert-low", "✅"
        elif pred_prob >= confidence_threshold:
            if any(k in threat_type for k in ["DDoS", "DoS", "Exploit", "APT", "Botnet", "Infiltration"]):
                threat_level, css_class, icon = "CRITICAL", "alert-critical", "🚨"
            else:
                threat_level, css_class, icon = "HIGH", "alert-high", "⚠️"
        else:
            threat_level, css_class, icon = "SUSPICIOUS", "alert-suspicious", "🔍"

        st.session_state.total_scans += 1
        if threat_level != "BENIGN": st.session_state.threat_count += 1

        # 4. Update Main Alert Box
        with alert_placeholder.container():
            st.markdown(f"""
                <div class='{css_class}'>
                    {icon} {threat_type} Detected<br>
                    <span style='font-size: 18px; font-weight: normal;'>Model Confidence: {pred_prob:.2f}% | Severity: {threat_level}</span>
                </div>
            """, unsafe_allow_html=True)

        # 5. Update DataFrames
        current_time = time.strftime("%H:%M:%S")
        unique_key = str(int(time.time() * 1000))

        log_entry = {"Timestamp": current_time, "Predicted": threat_type, "Actual": actual_label, "Severity": threat_level, "Confidence": f"{pred_prob:.2f}%"}
        st.session_state.history.insert(0, log_entry)
        st.session_state.history = st.session_state.history[:10]

        new_row = pd.DataFrame({'Time': [current_time], 'Confidence': [pred_prob], 'Threat': [threat_type]})
        st.session_state.chart_data = pd.concat([st.session_state.chart_data, new_row], ignore_index=True)
        if len(st.session_state.chart_data) > 30:
            st.session_state.chart_data = st.session_state.chart_data.iloc[1:]

        # 6. Update KPIs
        avg_conf = st.session_state.chart_data['Confidence'].mean()
        update_kpis(st.session_state.total_scans, st.session_state.threat_count, avg_conf)

        # 7. Render Charts (Premium Styling)
        # Area Chart
        fig_line = px.area(st.session_state.chart_data, x='Time', y='Confidence', color='Threat',
                           title="Live Confidence Area Chart", range_y=[0, 105])
        fig_line.update_layout(
            plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)',
            font=dict(color='#FAFAFA'), margin=dict(l=0, r=0, t=30, b=0), height=350,
            legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
        )
        fig_line.update_xaxes(showgrid=False)
        fig_line.update_yaxes(showgrid=True, gridcolor='rgba(255,255,255,0.1)')
        chart_placeholder.plotly_chart(fig_line, use_container_width=True, key=f"line_{unique_key}")

        # Donut Chart
        level_counts = pd.DataFrame(st.session_state.history)['Severity'].value_counts().reset_index()
        level_counts.columns = ['Severity', 'Count']
        fig_pie = px.pie(level_counts, names='Severity', values='Count', hole=0.6,
                         color='Severity', color_discrete_map={'BENIGN':'#38ef7d', 'CRITICAL':'#FF4B2B', 'HIGH':'#FFD200', 'SUSPICIOUS':'#4CA1AF'})
        fig_pie.update_layout(
            plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)',
            font=dict(color='#FAFAFA'), margin=dict(l=0, r=0, t=30, b=0), height=350, showlegend=True
        )
        fig_pie.update_traces(textposition='inside', textinfo='percent+label')
        pie_placeholder.plotly_chart(fig_pie, use_container_width=True, key=f"pie_{unique_key}")

        # Table
        with table_placeholder.container():
            st.dataframe(pd.DataFrame(st.session_state.history), use_container_width=True)

        time.sleep(1.5)

else:
    with alert_placeholder.container():
        st.info("👈 System on Standby. Click 'Initialize Scan' in the control panel to begin real-time monitoring.")

Overwriting app.py


In [ ]:
!pip install -q plotly

In [ ]:
import subprocess, time, threading, os
os.system("pkill -f streamlit")
threading.Thread(target=lambda: subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.enableCORS", "false"])).start()
time.sleep(5)
print("\n🔗 CLICK THE 'trycloudflare.com' LINK BELOW:\n")
!cloudflared tunnel --url http://localhost:8501


🔗 CLICK THE 'trycloudflare.com' LINK BELOW:

2026-11-19T18:03:23Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-11-19T18:03:23Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-11-19T18:03:25Z INF +--------------------------------------------------------------------------------------------+
2026-11-19T18:03:25Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-11-19T18:03:25Z INF |  https:/